In [3]:
!pip install -q -U transformers peft accelerate huggingface_hub gradio torch torchvision pillow "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 713.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
"""
Nepali Crop Disease Diagnosis — Demo Interface
Run this in Google Colab
or any machine with a GPU. Works on CPU too, just slower for the SmolVLM step.

Install first:
!pip install -q gradio transformers peft accelerate huggingface_hub torch torchvision pillow

Then just run this file (or paste into a Colab cell).
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import gradio as gr
from huggingface_hub import hf_hub_download
from transformers import AutoProcessor
try:
    from transformers import AutoModelForImageTextToText as AutoModelForVision2Seq
except ImportError:
    from transformers import AutoModelForVision2Seq
from peft import PeftModel

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_CLASSES = [
    "apple", "cherry", "grape", "maize", "peach",
    "pepper", "potato", "strawberry", "tomato",
]

CNN_REPO_ID = "prathamshrestha69/CNN-baseline"
CNN_FILENAME = "baseline_cnn.pt"
VLM_BASE_MODEL = "HuggingFaceTB/SmolVLM-256M-Instruct"
VLM_ADAPTER_REPO = "w4ashabii/SmolVLM256M_CropDisease"
IMG_SIZE = 128

VLM_QUESTION = "What crop is shown in this image, and does it have any disease? If so, name the disease."


# ---- Baseline CNN (same architecture as CNN-baseline.ipynb) ----
class BaselineCNN(nn.Module):
    def __init__(self, num_classes, img_size=128):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.3)
        feat_size = img_size // 8
        self.fc1 = nn.Linear(128 * feat_size * feat_size, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = torch.flatten(x, 1)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x)


print("Loading Baseline CNN...")
_cnn_weights_path = hf_hub_download(repo_id=CNN_REPO_ID, filename=CNN_FILENAME)
cnn_model = BaselineCNN(num_classes=len(MODEL_CLASSES), img_size=IMG_SIZE)
cnn_model.load_state_dict(torch.load(_cnn_weights_path, map_location=DEVICE))
cnn_model.to(DEVICE)
cnn_model.eval()

cnn_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

print("Loading SmolVLM + LoRA adapter...")
vlm_processor = AutoProcessor.from_pretrained(VLM_ADAPTER_REPO)
_vlm_base = AutoModelForVision2Seq.from_pretrained(
    VLM_BASE_MODEL,
    dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32,
)
vlm_model = PeftModel.from_pretrained(_vlm_base, VLM_ADAPTER_REPO)
vlm_model.to(DEVICE)
vlm_model.eval()

print("Both models loaded. Launching interface...")


def predict_cnn(image: Image.Image) -> str:
    tensor = cnn_transform(image.convert("RGB")).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = cnn_model(tensor)
        pred_idx = logits.argmax(dim=1).item()
        confidence = F.softmax(logits, dim=1)[0, pred_idx].item()
    crop = MODEL_CLASSES[pred_idx]
    return f"Crop: {crop.capitalize()}  (confidence: {confidence:.0%})\n(Baseline CNN only predicts crop type, not disease.)"


def predict_vlm(image: Image.Image) -> str:
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": VLM_QUESTION}]}]
    prompt = vlm_processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = vlm_processor(text=prompt, images=[image.convert("RGB")], return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        generated_ids = vlm_model.generate(**inputs, max_new_tokens=64)
    answer = vlm_processor.batch_decode(
        generated_ids[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )[0]
    return answer.strip()


def diagnose(image):
    if image is None:
        return "Please upload a leaf photo first.", "Please upload a leaf photo first."
    cnn_result = predict_cnn(image)
    vlm_result = predict_vlm(image)
    return cnn_result, vlm_result


with gr.Blocks(title="Nepali Crop Disease Diagnosis") as demo:
    gr.Markdown(
        """
        # Nepali Crop Disease Diagnosis — Demo
        Upload a photo of a crop leaf. Both models will analyse it below.

        **This is an assistive tool only.** Always confirm a diagnosis with an
        agricultural extension officer before making any treatment decision.
        """
    )
    with gr.Row():
        image_input = gr.Image(type="pil", label="Upload a leaf photo")
    with gr.Row():
        submit_btn = gr.Button("Diagnose", variant="primary")
    with gr.Row():
        cnn_output = gr.Textbox(label="Baseline CNN — Crop Classification", lines=3)
        vlm_output = gr.Textbox(label="SmolVLM (proposed model) — Crop + Disease", lines=3)

    submit_btn.click(fn=diagnose, inputs=image_input, outputs=[cnn_output, vlm_output])

demo.launch(share=True)

Loading Baseline CNN...
Loading SmolVLM + LoRA adapter...


Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 15.2MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Both models loaded. Launching interface...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8b6d001b22fb48ad81.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
